# Import packages

In [1]:
import pandas as pd
from scipy.io import arff

from utils.etl import train_test_split

In [2]:
random_state = 42

# Load data

Source: [ECG5000 Dataset link](https://www.timeseriesclassification.com/description.php?Dataset=ECG5000)

The dataset is composed of 5000 time series of ECG heartbeats. Each time series has 140 data points and 1 label.
The columns att1 to att140 represent each step of a time series and the target column represents the label: 1 to normal and 2-5 for anomalies.

Possible values for the label:
- 1: Normal
- 2: Left Bundle Branch Block (LBBB)
- 3: Right Bundle Branch Block (RBBB)
- 4: Premature Ventricular Contraction (PVC)
- 5: 5: Paced Beat (PB) 

In this step, the data will be loaded and concatenated.

In [3]:
# Load the ARFF files'
data1, meta1 = arff.loadarff('data/raw/ECG5000_TRAIN.arff')
data2, meta2 = arff.loadarff('data/raw/ECG5000_TEST.arff')

df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)

# Decode byte strings (ARFF often encodes strings as bytes)
df1 = df1.apply(lambda col: col.str.decode('utf-8') if col.dtype == object else col)
df2 = df2.apply(lambda col: col.str.decode('utf-8') if col.dtype == object else col)

# Concatenate
df = pd.concat([df1, df2], ignore_index=True)

In [4]:
df.columns

Index(['att1', 'att2', 'att3', 'att4', 'att5', 'att6', 'att7', 'att8', 'att9',
       'att10',
       ...
       'att132', 'att133', 'att134', 'att135', 'att136', 'att137', 'att138',
       'att139', 'att140', 'target'],
      dtype='str', length=141)

In [5]:
# Renaming 'att' columns
df = df.rename(columns={col: col.replace('att', 't')
      for col in df.columns
      if col.startswith('att')})

df.columns

Index(['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9', 't10',
       ...
       't132', 't133', 't134', 't135', 't136', 't137', 't138', 't139', 't140',
       'target'],
      dtype='str', length=141)

In [6]:
df['target'] = df['target'].astype(int)

df.head()

,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,...,t132,t133,t134,t135,t136,t137,t138,t139,t140,target
0,-0.112522,-2.827204,-3.773897,-4.349751,-4.376041,-3.474986,-2.181408,-1.818286,-1.250522,-0.477492,...,0.792168,0.933541,0.796958,0.578621,0.257740,0.228077,0.123431,0.925286,0.193137,1
1,-1.100878,-3.996840,-4.285843,-4.506579,-4.022377,-3.234368,-1.566126,-0.992258,-0.754680,0.042321,...,0.538356,0.656881,0.787490,0.724046,0.555784,0.476333,0.773820,1.119621,-1.436250,1
2,-0.567088,-2.593450,-3.874230,-4.584095,-4.187449,-3.151462,-1.742940,-1.490659,-1.183580,-0.394229,...,0.886073,0.531452,0.311377,-0.021919,-0.713683,-0.532197,0.321097,0.904227,-0.421797,1
3,0.490473,-1.914407,-3.616364,-4.318823,-4.268016,-3.881110,-2.993280,-1.671131,-1.333884,-0.965629,...,0.350816,0.499111,0.600345,0.842069,0.952074,0.990133,1.086798,1.403011,-0.383564,1
4,0.800232,-0.874252,-2.384761,-3.973292,-4.338224,-3.802422,-2.534510,-1.783423,-1.594450,-0.753199,...,1.148884,0.958434,1.059025,1.371682,1.277392,0.960304,0.971020,1.614392,1.421456,1


# Split data

Data will be splitted in these datasets:

- training (completely normal so that the autoencoder can learn the normal behaviour);
- hyperparameter tunning (completely normal);
- threshold tunning (mixed);
- testing (mixed).

In [7]:
normal_df = df.query('target == 1')
anomaly_df = df.query('target != 1')

print(f"Normal samples: {len(normal_df)}")
print(f"Anomaly samples: {len(anomaly_df)}")

Normal samples: 2919
Anomaly samples: 2081


In [8]:
# Normal df split
train_df, temp_df = train_test_split(
    normal_df, 
    test_size=0.30, 
    random_state=42
)

hyperparam_df, temp2_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    random_state=42
)

threshold_normal_df, test_normal_df = train_test_split(
    temp2_df, 
    test_size=0.33, 
    random_state=42
)

# Anomaly df split
threshold_anomaly_df, test_anomaly_df = train_test_split(
    anomaly_df, 
    test_size=0.50, 
    random_state=42
)

# Threshold e Test (mixed)
threshold_df = pd.concat([threshold_normal_df, threshold_anomaly_df], ignore_index=True).sample(frac=1, random_state=42)
test_df = pd.concat([test_normal_df, test_anomaly_df], ignore_index=True).sample(frac=1, random_state=42)

In [9]:
print(f"Train set: {train_df.shape}")
print(f"Hyperparameter set: {hyperparam_df.shape}")
print(f"Threshold set: {threshold_df.shape}")
print(f"Test set: {test_df.shape}")

Train set: (2044, 141)
Hyperparameter set: (438, 141)
Threshold set: (1334, 141)
Test set: (1184, 141)


As shown below, the train and hyperparameter tunning dataframes contain only normal samples

In [10]:
train_df['target'].value_counts()

target
1    2044
Name: count, dtype: int64

In [11]:
hyperparam_df['target'].value_counts()

target
1    438
Name: count, dtype: int64

# Save processed datasets

In [12]:
train_df.to_parquet('data/processed/train.parquet', index=False)
hyperparam_df.to_parquet('data/processed/hyperparam.parquet', index=False)
threshold_df.to_parquet('data/processed/threshold.parquet', index=False)
test_df.to_parquet('data/processed/test.parquet', index=False)